# Day 1: GDELT Feasibility Check

This notebook is a comparison reference for the DS-20 project. It does not train a sentiment model or make final project claims.

Day 1 questions:

- Does GDELT return enough Nigerian-source records?
- Are the records genuinely about food prices or cost of living?
- Do the records cover enough time and source domains?
- Is the result suitable for a manual sentiment-label sample?

## Project framing

The project framing, written before running the data cells:

- Problem: Nigerians feel the pressure of rising food prices and cost of living,
  but it is hard to see how that pressure shows up in news coverage or how the
  attention is changing over time.
- Research question: How has Nigerian online news attention to food prices and
  cost of living changed over the most recent available period, and how has the
  tone of that coverage (positive, neutral, negative) shifted over the same time?
- What this project will not measure: actual food prices, household purchasing
  behaviour, public opinion, or search interest. It tracks online news coverage
  only — and the volume figures are a Google News feed sample, not a complete
  record of all coverage. No causal claims will be made.

In [1]:
# Google Colab already includes these packages.
# For local Jupyter, install pandas and requests before running this cell.
from pathlib import Path
import json
import time

import pandas as pd
import requests
from IPython.display import display

In [2]:
# When running locally from the notebooks folder, the project root is its parent.
# In Colab, change PROJECT_DIR to the Drive folder created for this project.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DIR = PROJECT_DIR / 'data' / 'raw'
CLEAN_DIR = PROJECT_DIR / 'data' / 'clean'
print('Project root:', PROJECT_DIR.resolve())

Project root: C:\Users\brain\Documents\3mtt\3mtt_capstone_project


In [4]:
def fetch_gdelt(params, max_attempts=3, base_wait=30):
    endpoint = 'https://api.gdeltproject.org/api/v2/doc/doc'

    for attempt in range(1, max_attempts + 1):
        response = requests.get(
            endpoint,
            params=params,
            headers={'User-Agent': '3mtt-sentiment-tracker/1.0'},
            timeout=30,
        )

        if response.status_code == 200:
            return response.json()

        if response.status_code == 429 and attempt < max_attempts:
            wait_seconds = base_wait * attempt
            print(f'Rate limited on attempt {attempt}; waiting {wait_seconds} seconds.')
            time.sleep(wait_seconds)
            continue

        print(f'GDELT request failed with status {response.status_code}.')
        print(response.text[:500])
        return {}

    return {}

In [5]:
QUERY = 'sourcecountry:nigeria Nigeria (food OR "food prices" OR "cost of living" OR inflation OR rice OR bread)'

PARAMS = {
    'query': QUERY,
    'mode': 'artlist',
    'format': 'json',
    'maxrecords': 250,
    'timespan': '3months',
    'sort': 'datedesc',
}

payload = fetch_gdelt(PARAMS)
articles = pd.DataFrame(payload.get('articles', []))

print('Query:', QUERY)
print('Rows returned:', len(articles))
if not articles.empty:
    print('Fields:', list(articles.columns))
else:
    print('No articles were returned. Review the status message above.')

Rate limited on attempt 1; waiting 30 seconds.


ReadTimeout: HTTPSConnectionPool(host='api.gdeltproject.org', port=443): Read timed out. (read timeout=30)

## Contingency: Google News RSS

Use this only if the GDELT query fails the Day 1 feasibility gate. It is a media-search snapshot, not an official price dataset or a measure of public opinion. The final analysis must use one source, not a mixture.

In [7]:
USE_CONTINGENCY = True
RSS_URL = 'https://news.google.com/rss/search?q=Nigeria%20food%20prices%20cost%20of%20living&hl=en-NG&gl=NG&ceid=NG%3Aen'

if USE_CONTINGENCY:
    import xml.etree.ElementTree as ET

    rss_response = requests.get(
        RSS_URL,
        headers={'User-Agent': '3mtt-sentiment-tracker/1.0'},
        timeout=30,
    )
    rss_response.raise_for_status()
    root = ET.fromstring(rss_response.content)

    rss_rows = []
    for item in root.findall('./channel/item'):
        source_node = item.find('source')
        source = source_node.text if source_node is not None else ''
        raw_title = item.findtext('title', default='')
        source_suffix = f' - {source}'
        headline = raw_title[:-len(source_suffix)] if source and raw_title.endswith(source_suffix) else raw_title
        rss_rows.append({
            'url': item.findtext('link', default=''),
            'title': headline,
            'title_raw': raw_title,
            'pubDate': item.findtext('pubDate', default=''),
            'domain': source,
            'source_label': source,
            'sourcecountry': '',
            'language': '',
        })

    articles = pd.DataFrame(rss_rows)
    payload = {'articles': rss_rows}
    print('Google News RSS rows:', len(articles))
    print('Distinct source labels:', articles['domain'].nunique())
else:
    print('GDELT remains the active source. Set USE_CONTINGENCY = True only after the GDELT gate fails.')

Google News RSS rows: 100
Distinct source labels: 38


In [8]:
if not articles.empty:
    if 'seendate' in articles.columns:
        articles['seen_datetime'] = pd.to_datetime(
            articles['seendate'],
            format='%Y%m%dT%H%M%SZ',
            errors='coerce',
            utc=True,
        )
    elif 'pubDate' in articles.columns:
        articles['seen_datetime'] = pd.to_datetime(
            articles['pubDate'],
            errors='coerce',
            utc=True,
        )
    else:
        articles['seen_datetime'] = pd.NaT
    articles['week'] = articles['seen_datetime'].dt.tz_localize(None).dt.to_period('W').astype(str)

    print('Date range:', articles['seen_datetime'].min(), 'to', articles['seen_datetime'].max())
    print('Distinct domains:', articles['domain'].nunique())

    columns = ['title', 'seen_datetime', 'domain', 'sourcecountry', 'language']
    sample = articles[[column for column in columns if column in articles.columns]]
    display(sample.sample(min(20, len(sample)), random_state=42))

Date range: 2025-09-17 02:01:01+00:00 to 2026-08-10 20:30:08+00:00
Distinct domains: 38


,title,seen_datetime,domain,sourcecountry,language
83,Food price drop offers rare relief for families,2025-11-04 08:00:00+00:00,Business News Nigeria,,
53,Cost-of-Living Pressure Deepens as Fuel Prices...,2026-03-23 07:00:00+00:00,nigeriahousingmarket.com,,
70,"Households Struggle as ₦300,000 Income Falls S...",2026-04-16 07:00:00+00:00,nigeriahousingmarket.com,,
45,Need to address rising cost of living,2026-05-09 07:00:00+00:00,Daily Trust,,
44,Sallah: Consumers lament soaring food prices a...,2026-03-20 07:00:00+00:00,Daily Trust,,
39,"Same Groceries, Higher Bills",2026-03-17 07:00:00+00:00,dataphyte.com,,
22,"Reduce workers’ living costs, CPPE urges FG",2026-05-01 07:00:00+00:00,Punch Newspapers,,
80,CPI-Based Ranking: Most Affordable Nigerian St...,2026-02-17 19:20:07+00:00,nigeriahousingmarket.com,,
10,Nigeria’s Inflation To Rise On High Food Price...,2026-06-14 07:00:00+00:00,MarketForces Africa,,
0,Nigerians' cost of living pain deepens as elec...,2026-08-10 20:30:08+00:00,Reuters,,


In [9]:
TOPIC_TERMS = [
    'food',
    'price',
    'inflation',
    'cost of living',
    'affordability',
    'rice',
    'bread',
    'cooking oil',
    'scarcity',
]

def title_has_topic_term(title):
    title = str(title).lower()
    return any(term in title for term in TOPIC_TERMS)

if not articles.empty:
    articles['title_keyword_relevant'] = articles['title'].map(title_has_topic_term)
    relevant_count = int(articles['title_keyword_relevant'].sum())
    relevance_rate = articles['title_keyword_relevant'].mean()
    print('Headline keyword-relevant rows:', relevant_count)
    print(f'Headline keyword relevance rate: {relevance_rate:.1%}')
    print('This is a screening signal, not a final human relevance label.')
else:
    print('Relevance screening skipped because there are no articles.')

Headline keyword-relevant rows: 71
Headline keyword relevance rate: 71.0%
This is a screening signal, not a final human relevance label.


In [10]:
if not articles.empty:
    weekly_summary = (
        articles.groupby('week')
        .agg(
            article_count=('url', 'nunique'),
            distinct_domains=('domain', 'nunique'),
        )
        .reset_index()
        .sort_values('week')
    )
    display(weekly_summary)
else:
    weekly_summary = pd.DataFrame()
    print('Weekly summary skipped because there are no articles.')

,week,article_count,distinct_domains
0,2025-09-15/2025-09-21,2,2
1,2025-10-13/2025-10-19,1,1
2,2025-11-03/2025-11-09,1,1
3,2025-11-10/2025-11-16,1,1
4,2025-11-17/2025-11-23,1,1
5,2025-12-08/2025-12-14,1,1
6,2025-12-15/2025-12-21,2,1
7,2025-12-22/2025-12-28,2,2
8,2026-01-05/2026-01-11,1,1
9,2026-01-12/2026-01-18,1,1


In [11]:
# This is a provisional gate for deciding whether to proceed to final collection.
# It must be reviewed alongside the displayed headlines, not used blindly.
if not articles.empty:
    date_values = articles['seen_datetime'].dropna()
    date_span_days = (date_values.max() - date_values.min()).days if not date_values.empty else 0
    domain_count = articles['domain'].nunique()
    title_relevance_rate = articles['title_keyword_relevant'].mean()

    provisional_pass = (
        len(articles) >= 80
        and date_span_days >= 30
        and domain_count >= 5
        and title_relevance_rate >= 0.60
    )

    print('Date span in days:', date_span_days)
    print('Provisional status:', 'PASS TO FINAL COLLECTION' if provisional_pass else 'REVIEW QUERY OR USE CONTINGENCY')
    print('Important: manually inspect the sample before freezing data.')
else:
    print('Status: REVIEW QUERY OR USE CONTINGENCY')

Date span in days: 327
Provisional status: PASS TO FINAL COLLECTION
Important: manually inspect the sample before freezing data.


In [16]:
# Saving the Day 1 sample

# Keep this disabled until the query has passed manual review. The Day 1 sample is not the final frozen dataset.

SAVE_SAMPLE = True
if SAVE_SAMPLE and payload:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    source_name = 'google_news_rss' if USE_CONTINGENCY else 'gdelt'
    output_path = RAW_DIR / f'day1_{source_name}_sample.json'
    with open(output_path, 'w', encoding='utf-8') as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)
    print('Saved:', output_path)

Saved: C:\Users\brain\Documents\3mtt\3mtt_capstone_project\data\raw\day1_google_news_rss_sample.json


## Day 1 check-in

Recorded Day 1 feasibility values:

- Rows returned: 100 (Google News RSS contingency). GDELT failed repeatedly:
  HTTP 429 rate limits, then a read timeout on the final attempt.
- Date range : 17 Sep 2025 to 10 Aug 2026 in the feed sample.
- Final dataset window: 15 May 2026 to 11 Aug 2026 (3-month cutoff).
- Distinct domains: 38 in the Day-1 sample (51 in the final frozen set).
- Headline relevance rate: 71.0% (71 of 100 matched topic keywords -
  this is a screening signal, not a human label).
- Genuinely relevant headlines in the displayed sample: 18 of 20.
  2 peripheral: the Middle East war headline and the affordable-states ranking.
- Decision: USE CONTINGENCY. Google News RSS multi-query collection finalized;
  dataset frozen at 111 rows, 14 weeks, 51 sources
  (data/clean/articles_frozen.csv, 15 May - 11 Aug 2026).

Do not start sentiment labelling or model training until this decision is made.